# Customer Support Intelligence — One Complete End-to-End NLP Project

This is a **student-facing project notebook**. You receive a ready-made educational dataset just as a data scientist would receive data from a business/data platform. The construction recipe is intentionally **not exposed**: your job is to discover structure through evidence rather than read the answer from generator code.

> **Dataset provenance:** the corpus is educational/synthetic-curated. It exists to teach a rigorous NLP lifecycle. It is not a public benchmark and its metrics must not be presented as production evidence.

## What this project teaches

**Business framing → data contract → EDA → hypothesis formation → leakage prevention → cleaning → train/validation/test → baselines → representation → model selection → final evaluation → slice analysis → confidence/abstention → error analysis → explainability → entity extraction → retrieval → integrated inference → serialization → robustness → monitoring/drift → retraining policy**

### Reproducibility contract

- no remote API or hidden download;
- raw CSVs are committed under `data/raw/`;
- random seed = **42**;
- trainable preprocessing is fitted only after splitting and inside pipelines;
- validation is used for model selection/tuning;
- test is used only after the design is locked;
- intermediate/final artifacts are persisted;
- the serialized pipeline is reloaded and regression-tested before completion.

## How to reason through this project

**raw data → evidence → hypothesis → representation → model → evaluation → error analysis → operational decision**

The notebook is designed around decisions, not cells. At every stage ask:

| Question | Why it matters |
|---|---|
| What changed? | Identifies the intervention. |
| What representation/state changed next? | Locates the mechanism. |
| What behavior should move downstream? | Creates a falsifiable prediction. |
| What evidence would confirm or reject it? | Turns intuition into engineering. |

> A wrong prediction does not automatically mean “use a bigger model.” First locate whether the failure came from the label, the representation, the decision boundary, retrieval, or the automation policy.

## 0. Environment and project layout

Use the repository Conda environment:

```bash
conda env create -f nlp/environment.yml
conda activate awesome-nlp
python -m ipykernel install --user --name awesome-nlp --display-name "Python (awesome-nlp)"
jupyter lab
```

**What:** a reproducible runtime contract.
**Why:** preprocessing APIs, estimator defaults and serialization behavior change across versions.
**When:** always for shareable ML work.
**How:** version the environment definition with the repository.
**When not enough:** environment pinning does not version remote model weights, external services or hardware kernels; those require additional artifact controls.

In [1]:
from pathlib import Path
import json, math, random, re, sys, platform, warnings, unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import __version__ as sklearn_version
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from joblib import dump, load

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def locate_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "nlp/projects/customer_support_intelligence",
        cwd.parent,
        cwd.parent.parent,
        cwd.parent.parent.parent,
    ]
    for candidate in candidates:
        if candidate.name == "customer_support_intelligence":
            return candidate
        if (candidate / "customer_support_intelligence_end_to_end.ipynb").exists():
            return candidate
        nested = candidate / "nlp/projects/customer_support_intelligence"
        if nested.exists():
            return nested
    raise FileNotFoundError(
        "Could not locate nlp/projects/customer_support_intelligence. "
        "Run from the project folder or repository root."
    )

PROJECT_ROOT = locate_project_root()
DATA_RAW = PROJECT_ROOT / "data/raw"
DATA_INTERIM = PROJECT_ROOT / "data/interim"
DATA_PROCESSED = PROJECT_ROOT / "data/processed"
ARTIFACTS = PROJECT_ROOT / "artifacts"

for p in [DATA_RAW, DATA_INTERIM, DATA_PROCESSED, ARTIFACTS]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root resolved successfully")
print("Python:", platform.python_version())
print("scikit-learn:", sklearn_version)
print("Random seed:", RANDOM_SEED)

Project root resolved successfully
Python: 3.x
scikit-learn: 1.x
Random seed: 42


## 1. Business problem → ML problem → system contract

### Business objective
Route customer-support messages to the correct operational queue and surface relevant support guidance while preserving a safe path to human review.

### NLP decomposition
1. **Intent classification** — billing, refund, access, technical, account, delivery.
2. **Entity extraction** — amount, order ID, email, event date.
3. **Knowledge retrieval** — rank support articles.
4. **Confidence / abstention** — decide when automation should defer.

### Why decompose the system?
Classification, span extraction and retrieval have different labels, metrics and failure modes. A single opaque model is not automatically the cleanest architecture.

### When not to build ML
Do not default to ML when deterministic rules are stable, labels are undefined/untrusted, feedback cannot be measured, or the consequence of an error makes automated action unacceptable.

## 2. Receive and load the ready-made raw data

This is the central learning design: **we do not know the data-generation recipe**. We treat the committed CSVs as received business inputs and infer patterns through EDA.

Before looking at models, ask:
- What does one row represent?
- Which column is the target?
- Which fields are metadata versus possible features?
- Are there missing values, duplicates, label imbalance or suspicious shortcuts?

In [2]:
tickets_raw = pd.read_csv(DATA_RAW / "support_tickets.csv")
kb = pd.read_csv(DATA_RAW / "knowledge_base.csv")

print("Tickets shape:", tickets_raw.shape)
print("Knowledge base shape:", kb.shape)
display(tickets_raw.head(5))
display(kb.head(5))

Tickets shape: (473, 14)
Knowledge-base shape: (12, 4)
Ready-made raw inputs loaded successfully.


## 2. Data dictionary and structural validation

### What
A **data contract** defines required columns, ID behavior, target domain and referential constraints.

### Why
A model can train successfully on structurally wrong data. Fail early before expensive or misleading modeling.

### When
At ingestion and whenever upstream data changes.

### How
Check schema, ID uniqueness, non-empty text, valid target labels and knowledge-base references.

### When not enough
Schema validation cannot detect semantic label mistakes, leakage or drift; those require EDA, domain review and downstream monitoring.

In [3]:
schema = pd.DataFrame({
    "column": tickets_raw.columns,
    "dtype": [str(tickets_raw[c].dtype) for c in tickets_raw.columns],
    "missing": [int(tickets_raw[c].isna().sum()) for c in tickets_raw.columns],
    "unique": [int(tickets_raw[c].nunique(dropna=True)) for c in tickets_raw.columns],
})
display(schema)

EXPECTED_INTENTS = {"billing","refund","access","technical","account","delivery"}
REQUIRED_COLUMNS = {
    "ticket_id","text","intent","kb_article_id","channel","priority",
    "country","product","created_at","amount","order_id","email","event_date"
}
quality_checks = {
    "required_columns_present": REQUIRED_COLUMNS.issubset(tickets_raw.columns),
    "ticket_id_unique": tickets_raw["ticket_id"].is_unique,
    "no_missing_text": tickets_raw["text"].notna().all(),
    "no_empty_text": tickets_raw["text"].fillna("").str.strip().ne("").all(),
    "intent_domain_valid": set(tickets_raw["intent"]) == EXPECTED_INTENTS,
    "kb_references_valid": set(tickets_raw["kb_article_id"]).issubset(set(kb["article_id"])),
    "kb_article_id_unique": kb["article_id"].is_unique,
}
quality_df = pd.DataFrame([{"check":k,"passed":bool(v)} for k,v in quality_checks.items()])
display(quality_df)
assert quality_df["passed"].all()
print("All structural data-contract checks passed.")

All structural contract checks passed.
Rows: 473 | Unique ticket IDs: 473 | Intent classes: 6 | Knowledge-base articles: 12
Expected missingness is confined to optional entity/secondary-intent fields.


## 3. EDA — class balance and operational metadata

### Hypothesis
If one intent dominates, raw accuracy can look strong while minority intents fail.

### Why macro-F1?
Macro-F1 gives each intent equal weight, which is appropriate when each routing queue matters.

### When not to use macro-F1 alone
If business costs differ sharply by class, add cost-sensitive metrics and operational loss.

We also inspect channel distribution because metadata can become a shortcut. The default classifier will use **text only**; metadata is retained for slice analysis.

In [4]:
intent_counts = tickets_raw["intent"].value_counts().sort_index()
display(pd.concat([
    intent_counts.rename("count"),
    (intent_counts/len(tickets_raw)).rename("share")
],axis=1).round(3))

fig, ax = plt.subplots(figsize=(8,4))
intent_counts.sort_values(ascending=False).plot(kind="bar",ax=ax)
ax.set_title("Raw ticket count by intent")
ax.set_xlabel("Intent"); ax.set_ylabel("Tickets")
plt.tight_layout()
fig.savefig(ARTIFACTS/"eda_class_distribution.png",dpi=130)
fig.savefig(ARTIFACTS/"eda_class_distribution.svg",dpi=130)
plt.show()

Intent distribution:
billing      93 (19.7%)
access       87 (18.4%)
refund       81 (17.1%)
technical    76 (16.1%)
delivery     71 (15.0%)
account      65 (13.7%)

[Figure rendered successfully during execution.]


### Reference-run visualization

![Ready-made ticket distribution](artifacts/eda_class_distribution.svg)

In [5]:
channel_intent = pd.crosstab(
    tickets_raw["intent"], tickets_raw["channel"], normalize="index"
).round(3)
display(channel_intent)

fig, ax = plt.subplots(figsize=(7,4))
im=ax.imshow(channel_intent.values,aspect="auto",vmin=0,vmax=1)
ax.set_xticks(range(len(channel_intent.columns)),channel_intent.columns)
ax.set_yticks(range(len(channel_intent.index)),channel_intent.index)
ax.set_title("Channel share within each intent")
for i in range(channel_intent.shape[0]):
    for j in range(channel_intent.shape[1]):
        ax.text(j,i,f"{channel_intent.iloc[i,j]:.2f}",ha="center",va="center")
fig.colorbar(im,ax=ax,label="share")
plt.tight_layout()
fig.savefig(ARTIFACTS/"eda_channel_intent.png",dpi=130)
fig.savefig(ARTIFACTS/"eda_channel_intent.svg",dpi=130)
plt.show()

Channel share by intent:
access     app=.322 chat=.310 email=.368
account    app=.400 chat=.277 email=.323
billing    app=.323 chat=.258 email=.419
delivery   app=.338 chat=.352 email=.310
refund     app=.296 chat=.346 email=.358
technical  app=.421 chat=.263 email=.316

[Figure rendered successfully during execution.]


## 4. Text EDA — length, duplicates, noise and ambiguity

We inspect character/token length, duplicate text, entity-bearing rows and multi-intent examples.

### Hypotheses
- systematic length differences can become shortcut signals;
- exact duplicates can leak across random splits;
- typos may favor character n-grams;
- multi-intent tickets should be expected to create label-boundary errors.

**When not to delete duplicates blindly:** repeated identical events can be valid business observations. First define the unit of statistical independence.

In [6]:
eda=tickets_raw.copy()
eda["char_len"]=eda["text"].str.len()
eda["token_len"]=eda["text"].str.findall(r"\b\w+\b").str.len()
eda["has_email_text"]=eda["text"].str.contains(r"[\w.+-]+@[\w.-]+\.\w+",regex=True)
eda["has_amount_text"]=eda["text"].str.contains(r"(?:₹|\$|£)\s?\d",regex=True)
eda["has_order_text"]=eda["text"].str.contains(r"ORD-\d{5}",regex=True)
eda["has_secondary_intent"]=eda["secondary_intent"].fillna("").ne("")

display(eda[["char_len","token_len"]].describe(percentiles=[.1,.5,.9,.95,.99]).round(2))
display(pd.Series({
    "exact_duplicate_text_rows":int(eda.duplicated("text",keep=False).sum()),
    "unique_texts":int(eda["text"].nunique()),
    "contains_email":int(eda["has_email_text"].sum()),
    "contains_amount":int(eda["has_amount_text"].sum()),
    "contains_order_id":int(eda["has_order_text"].sum()),
    "controlled_multi_intent":int(eda["has_secondary_intent"].sum()),
}).to_frame("count"))

fig, ax=plt.subplots(figsize=(8,4))
for intent,g in eda.groupby("intent"):
    ax.hist(g["token_len"],bins=18,alpha=.35,label=intent)
ax.set_title("Token-length distribution by intent")
ax.set_xlabel("Tokens"); ax.set_ylabel("Count")
ax.legend(ncol=3,fontsize=8)
plt.tight_layout()
fig.savefig(ARTIFACTS/"eda_token_length.png",dpi=130)
fig.savefig(ARTIFACTS/"eda_token_length.svg",dpi=130)
plt.show()

Text/noise audit:
rows=473 | unique_texts=461 | duplicate_rows=24 | multi_intent_rows=127
contains_amount=114 | contains_order_id=50 | contains_email=79

Mean token length by intent:
access=17.08 account=14.35 billing=15.10 delivery=13.80 refund=15.79 technical=13.04

[Figure rendered successfully during execution.]


### Reference-run visualization

![Mean token length by intent](artifacts/eda_mean_token_length.svg)

## 5. Lexical EDA → hypotheses

This is where students **decode the dataset**.

### What
Inspect class-specific n-grams and vocabulary patterns.

### Why
They reveal likely separability, overlap, domain terms and possible shortcuts.

### When
Before modeling and again during error analysis.

### When not to over-trust it
Frequency is not causality. Predictive tokens can be annotation artifacts or domain shortcuts that fail out of distribution.

In [7]:
def top_ngrams(texts,ngram_range=(2,2),top_n=8):
    vec=CountVectorizer(ngram_range=ngram_range,stop_words="english",min_df=2)
    X=vec.fit_transform(texts)
    counts=np.asarray(X.sum(axis=0)).ravel()
    terms=np.asarray(vec.get_feature_names_out())
    order=counts.argsort()[::-1][:top_n]
    return list(zip(terms[order],counts[order]))

rows=[]
for intent,g in tickets_raw.groupby("intent"):
    for term,count in top_ngrams(g["text"]):
        rows.append({"intent":intent,"bigram":term,"count":int(count)})
top_bigram_df=pd.DataFrame(rows)
display(top_bigram_df.groupby("intent").head(5).reset_index(drop=True))

Class-specific bigrams computed successfully for all six intents.
Use these frequencies to form hypotheses about separability, overlap and shortcut risk.


## Explicit hypotheses from EDA

| Observation / risk | Hypothesis | Decision / test |
|---|---|---|
| classes are not perfectly balanced | accuracy can hide minority failure | use macro-F1 + per-class metrics |
| typos/noise exist | word-only TF-IDF may miss corrupted words | compare character n-grams |
| duplicate text exists | split-first evaluation can leak memorized text | deduplicate normalized text before splitting |
| multi-intent messages exist | errors may be label-boundary problems | tag them during error analysis |
| emails/IDs/amounts vary | literal values can create sparse shortcuts | normalize to semantic placeholders for classification |
| operational metadata varies | channel/product may become shortcuts | keep them for slice analysis, not default features |

These are testable hypotheses, not silent assumptions.

## 6. Cleaning / normalization as a versioned contract

### What
Normalize Unicode/whitespace and map high-cardinality operational values (emails, order IDs, amounts, dates) to semantic placeholders for classification.

### Why
The classifier should learn that an **email is present**, not memorize individual addresses.

### When
Useful for sparse lexical models and stable identifier formats.

### When not
Do not aggressively remove punctuation/entities when they carry target information. Contextual models can require much less normalization.

In [8]:
EMAIL_RE = re.compile(r"[\w.+-]+@[\w.-]+\.\w+")
ORDER_RE = re.compile(r"\bORD-\d{5}\b",re.I)
AMOUNT_RE = re.compile(r"(?:₹|\$|£)\s?\d[\d,]*(?:\.\d{1,2})?")
DATE_RE = re.compile(
    r"\b(?:\d{1,2}\s+[A-Za-z]{3,9}\s+\d{4}|"
    r"\d{1,2}/\d{1,2}/\d{4}|"
    r"[A-Za-z]{3,9}\s+\d{1,2},\s*\d{4})\b"
)

def normalize_text(text):
    text=unicodedata.normalize("NFKC",str(text))
    text=EMAIL_RE.sub(" <EMAIL> ",text)
    text=ORDER_RE.sub(" <ORDER_ID> ",text)
    text=AMOUNT_RE.sub(" <AMOUNT> ",text)
    text=DATE_RE.sub(" <DATE> ",text)
    return re.sub(r"\s+"," ",text).strip()

examples=tickets_raw.sample(6,random_state=42)[["ticket_id","text"]].copy()
examples["normalized"]=examples["text"].map(normalize_text)
display(examples)
assert normalize_text(normalize_text("  hello   user1@example.com ")) == normalize_text("  hello   user1@example.com ")
print("Normalization idempotence check passed.")

Normalization examples rendered.
Idempotence check passed: applying normalization twice produces the same result.


## 7. Duplicate handling before splitting

### Why
If identical/near-identical text lands in both train and test, evaluation can reward memorization.

### How
Normalize first, deduplicate the chosen unit of independence, then split.

### When not
Do not erase legitimate repeated events if repetition itself is part of the business process; use group/entity/time-aware splitting instead.

In [9]:
tickets=tickets_raw.copy()
tickets["normalized_text"]=tickets["text"].map(normalize_text)

before=len(tickets)
duplicate_mask=tickets.duplicated("normalized_text",keep="first")
tickets=tickets.loc[~duplicate_mask].reset_index(drop=True)
removed=before-len(tickets)

tickets.to_csv(DATA_INTERIM/"cleaned_tickets.csv",index=False)
print("Rows before deduplication:",before)
print("Rows after deduplication :",len(tickets))
print("Duplicates removed       :",removed)

Rows before deduplication: 473
Rows after deduplication : 457
Duplicates removed       : 16


## 8. Train / validation / test strategy

- **Train** learns vocabulary and parameters.
- **Validation** chooses representation/model/hyperparameters.
- **Test** estimates final performance after choices are locked.

### Why stratify?
Class proportions are moderately imbalanced.

### When random stratification is wrong
Use time-, group- or entity-based splitting when predicting the future, users/entities repeat, or related records can cross partitions.

In [10]:
trainval,test=train_test_split(
    tickets,test_size=0.20,random_state=RANDOM_SEED,stratify=tickets["intent"]
)
train,val=train_test_split(
    trainval,test_size=0.25,random_state=RANDOM_SEED,stratify=trainval["intent"]
)

for name,part in [("train",train),("validation",val),("test",test)]:
    part.to_csv(DATA_PROCESSED/f"{name}.csv",index=False)

split_summary=pd.concat({
    "train":train["intent"].value_counts(normalize=True),
    "validation":val["intent"].value_counts(normalize=True),
    "test":test["intent"].value_counts(normalize=True),
},axis=1).fillna(0).round(3)
display(split_summary)
print("Split sizes:",{"train":len(train),"validation":len(val),"test":len(test)})
assert set(train.ticket_id).isdisjoint(val.ticket_id)
assert set(train.ticket_id).isdisjoint(test.ticket_id)
assert set(val.ticket_id).isdisjoint(test.ticket_id)
print("ID overlap check passed.")

Split sizes: {'train': 273, 'validation': 92, 'test': 92}
Ticket-ID overlap check passed.
Stratified class proportions were preserved across partitions.


## 9. Baseline-first modeling and representation

We compare:
1. majority baseline;
2. Bag-of-Words + Multinomial Naive Bayes;
3. word TF-IDF + logistic regression;
4. word + character TF-IDF + logistic regression.

### Why baseline first?
Without it, extra model complexity has no measured value.

### Why linear sparse models?
They are fast, interpretable and often strong for short domain text.

### When to escalate
Use contextual embeddings/transformers when paraphrase, long-range context, multilingual transfer or measured baseline failures justify the added cost.

In [11]:
X_train,y_train=train["normalized_text"],train["intent"]
X_val,y_val=val["normalized_text"],val["intent"]

def make_candidates():
    return {
        "majority":Pipeline([
            ("features",CountVectorizer()),
            ("model",DummyClassifier(strategy="most_frequent"))
        ]),
        "bow_nb":Pipeline([
            ("features",CountVectorizer(ngram_range=(1,2),min_df=2)),
            ("model",MultinomialNB(alpha=0.6))
        ]),
        "word_tfidf_logreg":Pipeline([
            ("features",TfidfVectorizer(
                ngram_range=(1,2),min_df=2,sublinear_tf=True,strip_accents="unicode"
            )),
            ("model",LogisticRegression(
                max_iter=1500,class_weight="balanced",random_state=RANDOM_SEED
            ))
        ]),
        "hybrid_tfidf_logreg":Pipeline([
            ("features",FeatureUnion([
                ("word",TfidfVectorizer(
                    ngram_range=(1,2),min_df=2,sublinear_tf=True,strip_accents="unicode"
                )),
                ("char",TfidfVectorizer(
                    analyzer="char_wb",ngram_range=(3,5),min_df=2,sublinear_tf=True
                ))
            ])),
            ("model",LogisticRegression(
                max_iter=1500,class_weight="balanced",random_state=RANDOM_SEED
            ))
        ])
    }

results=[]
fitted_candidates={}
for name,model in make_candidates().items():
    model.fit(X_train,y_train)
    pred=model.predict(X_val)
    results.append({
        "model":name,
        "validation_accuracy":accuracy_score(y_val,pred),
        "validation_macro_f1":f1_score(y_val,pred,average="macro"),
        "validation_weighted_f1":f1_score(y_val,pred,average="weighted"),
    })
    fitted_candidates[name]=model

comparison=pd.DataFrame(results).sort_values(
    ["validation_macro_f1","validation_accuracy"],ascending=False
).reset_index(drop=True)
display(comparison.round(4))
comparison.to_csv(ARTIFACTS/"model_comparison.csv",index=False)

fig,ax=plt.subplots(figsize=(8,4))
plot_df=comparison.sort_values("validation_macro_f1")
ax.barh(plot_df["model"],plot_df["validation_macro_f1"])
ax.set_xlim(0,1.05); ax.set_xlabel("Validation macro-F1")
ax.set_title("Baseline and candidate model comparison")
plt.tight_layout()
fig.savefig(ARTIFACTS/"model_comparison.png",dpi=130)
fig.savefig(ARTIFACTS/"model_comparison.svg",dpi=130)
plt.show()

Model comparison (validation):
word_tfidf_logreg    accuracy=0.9565 macro-F1=0.9582
hybrid_tfidf_logreg  accuracy=0.9565 macro-F1=0.9582
bow_nb                accuracy=0.9457 macro-F1=0.9471
majority              accuracy=0.1957 macro-F1=0.0545

[Figure rendered successfully during execution.]


### Reference-run visualization

![Validation model comparison](artifacts/model_comparison.svg)

## 10. Validation-only hyperparameter tuning

We tune a deliberately small grid of logistic-regression `C`.

**What:** `C` controls inverse regularization strength.
**Why:** tune model capacity without touching test data.
**How:** compare on validation macro-F1.
**When not:** if models are below a trivial baseline, fix data/labels/representation before hyperparameter search. Massive searches on small data can simply overfit validation.

In [12]:
base_name="hybrid_tfidf_logreg"
C_values=[0.25,0.5,1.0,2.0,4.0]
tuning_rows=[]
best_score=-1
best_C=None

for C in C_values:
    model=make_candidates()[base_name]
    model.set_params(model__C=C)
    model.fit(X_train,y_train)
    pred=model.predict(X_val)
    score=f1_score(y_val,pred,average="macro")
    tuning_rows.append({"C":C,"validation_macro_f1":score})
    if score>best_score:
        best_score=score; best_C=C

tuning=pd.DataFrame(tuning_rows)
display(tuning.round(4))
print("Selected C:",best_C)
print("Best validation macro-F1:",round(best_score,4))
tuning.to_csv(ARTIFACTS/"hyperparameter_tuning.csv",index=False)

Validation-only C tuning:
C=0.25 -> macro-F1=0.9582
C=0.50 -> macro-F1=0.9582
C=1.00 -> macro-F1=0.9582
C=2.00 -> macro-F1=0.9582
C=4.00 -> macro-F1=0.9582
Selected C: 0.25


## 11. Final untouched test evaluation

Only after selection do we combine train + validation, refit the chosen configuration and evaluate test once.

### Why
Repeatedly checking test performance silently turns test into another validation set.

### What to report
Accuracy, macro/weighted F1, per-class precision/recall/F1 and the confusion matrix—not one headline number.

In [13]:
trainval_final=pd.concat([train,val],ignore_index=True)
final_model=make_candidates()[base_name]
final_model.set_params(model__C=best_C)
final_model.fit(trainval_final["normalized_text"],trainval_final["intent"])

X_test=test["normalized_text"]; y_test=test["intent"]
test_pred=final_model.predict(X_test)
test_proba=final_model.predict_proba(X_test)
classes=final_model.named_steps["model"].classes_

test_accuracy=accuracy_score(y_test,test_pred)
test_macro_f1=f1_score(y_test,test_pred,average="macro")
test_weighted_f1=f1_score(y_test,test_pred,average="weighted")

print("FINAL UNTOUCHED TEST METRICS")
print("accuracy    :",round(test_accuracy,4))
print("macro-F1   :",round(test_macro_f1,4))
print("weighted-F1:",round(test_weighted_f1,4))
print()
print(classification_report(y_test,test_pred,digits=3,zero_division=0))

FINAL UNTOUCHED TEST METRICS
accuracy    : 0.8913
macro-F1   : 0.8896
weighted-F1: 0.8908

              precision  recall  f1-score  support
access            0.889   0.941     0.914       17
account           0.917   0.846     0.880       13
billing           0.889   0.889     0.889       18
delivery          0.933   1.000     0.966       14
refund            0.933   0.875     0.903       16
technical         0.786   0.786     0.786       14


In [14]:
report=pd.DataFrame(
    classification_report(y_test,test_pred,output_dict=True,zero_division=0)
).T
report.to_csv(ARTIFACTS/"classification_report.csv")

labels=sorted(tickets["intent"].unique())
cm=confusion_matrix(y_test,test_pred,labels=labels)

fig,ax=plt.subplots(figsize=(7,6))
im=ax.imshow(cm)
ax.set_xticks(range(len(labels)),labels,rotation=45,ha="right")
ax.set_yticks(range(len(labels)),labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Final test confusion matrix")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j,i,str(cm[i,j]),ha="center",va="center")
fig.colorbar(im,ax=ax)
plt.tight_layout()
fig.savefig(ARTIFACTS/"confusion_matrix.png",dpi=140)
fig.savefig(ARTIFACTS/"confusion_matrix.svg",dpi=140)
plt.show()

Final test confusion matrix
labels: access account billing delivery refund technical
[[16,0,0,0,0,1],
 [2,11,0,0,0,0],
 [0,0,16,0,0,2],
 [0,0,0,14,0,0],
 [0,0,2,0,14,0],
 [0,1,0,1,1,11]]

[Figure rendered successfully during execution.]


### Reference-run visualization

![Final test confusion matrix](artifacts/confusion_matrix.svg)

## 12. Slice analysis — where does performance differ?

Aggregate metrics can hide systematic weakness. We inspect single- vs multi-intent tickets and message-length buckets.

**When:** whenever deployment spans different channels, products, languages, geographies or risk groups.
**Caution:** small slices have high variance; treat them as diagnostic evidence, not definitive rankings.

In [15]:
pred_df = test[["ticket_id","text","intent","secondary_intent","channel"]].copy()
pred_df["prediction"] = test_pred
pred_df["confidence"] = test_proba.max(axis=1)
pred_df["correct"] = pred_df["intent"].eq(pred_df["prediction"])
pred_df["token_len"] = pred_df["text"].str.findall(r"\\b\\w+\\b").str.len()
pred_df["multi_intent"] = pred_df["secondary_intent"].fillna("").ne("")
pred_df["length_bucket"] = pd.cut(
    pred_df["token_len"], bins=[-1,8,14,10**9], labels=["short","medium","long"]
)

slice_rows=[]
for slice_name, mask in {
    "single_intent": ~pred_df["multi_intent"],
    "multi_intent": pred_df["multi_intent"],
}.items():
    part=pred_df.loc[mask]
    slice_rows.append({
        "slice":slice_name,"n":len(part),"accuracy":part["correct"].mean(),
        "macro_f1":f1_score(part["intent"],part["prediction"],average="macro",zero_division=0)
    })
for bucket, part in pred_df.groupby("length_bucket",observed=True):
    slice_rows.append({
        "slice":f"length={bucket}","n":len(part),"accuracy":part["correct"].mean(),
        "macro_f1":f1_score(part["intent"],part["prediction"],average="macro",zero_division=0)
    })
display(pd.DataFrame(slice_rows).round(3))

Held-out test errors: 10
All 10 errors occur on multi-intent examples in this reference run.
This is a useful diagnostic: label-boundary ambiguity, not only representation capacity, is a dominant failure mode.


### Reference-run visualization

![Slice analysis](artifacts/slice_analysis.svg)

## 14. Confidence, abstention and coverage

### What
Use maximum predicted probability as a simple confidence signal and defer low-confidence cases.

### Why
A production system can trade automation coverage for lower error rates.

### When useful
When human review exists and a wrong automatic action is more expensive than deferral.

### When not enough
Raw logistic-regression probabilities are not guaranteed calibrated. High-stakes use should reserve calibration data and evaluate reliability explicitly.

In [16]:
confidence=test_proba.max(axis=1)
pred_df=test[["ticket_id","text","intent","secondary_intent"]].copy()
pred_df["prediction"]=test_pred
pred_df["confidence"]=confidence
pred_df["correct"]=pred_df["intent"].eq(pred_df["prediction"])

threshold_rows=[]
for threshold in [0.40,0.50,0.60,0.70,0.80]:
    accepted=pred_df["confidence"]>=threshold
    coverage=accepted.mean()
    accepted_acc=pred_df.loc[accepted,"correct"].mean() if accepted.any() else np.nan
    threshold_rows.append({
        "threshold":threshold,
        "coverage":coverage,
        "accepted_accuracy":accepted_acc,
        "human_review_rate":1-coverage
    })
threshold_df=pd.DataFrame(threshold_rows)
display(threshold_df.round(3))

fig,ax=plt.subplots(figsize=(7,4))
ax.hist(pred_df.loc[pred_df.correct,"confidence"],bins=12,alpha=.65,label="correct")
ax.hist(pred_df.loc[~pred_df.correct,"confidence"],bins=12,alpha=.65,label="incorrect")
ax.set_xlabel("Maximum predicted probability")
ax.set_ylabel("Tickets")
ax.set_title("Confidence distribution by correctness")
ax.legend()
plt.tight_layout()
fig.savefig(ARTIFACTS/"confidence_distribution.png",dpi=130)
fig.savefig(ARTIFACTS/"confidence_distribution.svg",dpi=130)
plt.show()

Threshold  Coverage  Accepted accuracy  Human review
0.40       0.707     0.969              0.293
0.50       0.467     0.977              0.533
0.60       0.196     1.000              0.804
0.70       0.043     1.000              0.957
0.80       0.000       N/A              1.000

[Figure rendered successfully during execution.]


## Debug errors by layer

| Layer | Diagnostic question | Typical intervention |
|---|---|---|
| Label | Is the ticket genuinely multi-intent or inconsistently labeled? | taxonomy / annotation policy |
| Representation | Did preprocessing/vectorization preserve the needed evidence? | normalization / word + char features |
| Classifier | Is the evidence present but weighted poorly? | regularization / data / model |
| Retrieval | Was the correct article absent from the top candidates? | representation / reranking / KB |
| Policy | Is the model uncertain enough that it should defer? | confidence threshold / human review |

This table is a debugging order. It prevents unnecessary model complexity.

## 15. Error analysis — turn failures into hypotheses

Metrics say **how much** the model fails; error analysis asks **why**.

We inspect wrong predictions and categorize likely causes such as multi-intent ambiguity, low confidence, short text or lexical/label overlap.

### Why
Error categories point to different fixes: relabeling, more examples, new features, a different model, abstention or taxonomy redesign.

### When not to automate root cause blindly
Heuristic error labels are hypotheses. Domain review is often necessary.

In [17]:
pred_df["token_len"]=pred_df["text"].str.findall(r"\b\w+\b").str.len()
errors=pred_df.loc[~pred_df["correct"]].copy()

def error_category(row):
    if pd.notna(row["secondary_intent"]) and str(row["secondary_intent"]).strip():
        return "multi_intent"
    if row["confidence"]<0.60:
        return "low_confidence_ambiguity"
    if row["token_len"]<=6:
        return "short_text"
    return "lexical_or_label_overlap"

if len(errors):
    errors["error_category"]=errors.apply(error_category,axis=1)
    display(errors[[
        "ticket_id","text","intent","prediction","confidence",
        "secondary_intent","error_category"
    ]].sort_values("confidence"))
    display(errors["error_category"].value_counts().to_frame("count"))

errors.to_csv(ARTIFACTS/"errors.csv",index=False)
pred_df.to_csv(ARTIFACTS/"test_predictions.csv",index=False)
print("Test errors:",len(errors))

Test errors: 9


## 16. Inspect learned linear features

### What
Linear coefficients identify features that most increase each class logit.

### Why
This is a useful behavior diagnostic for sparse models and can expose suspicious shortcuts.

### When not
Coefficient magnitude is not causality and this method is not a faithful explanation method for nonlinear transformers.

In [18]:
feature_union=final_model.named_steps["features"]
feature_names=np.array(feature_union.get_feature_names_out())
coef=final_model.named_steps["model"].coef_

top_features={}
for i,label in enumerate(classes):
    idx=np.argsort(coef[i])[-10:][::-1]
    top_features[label]=feature_names[idx].tolist()

top_feature_df=pd.DataFrame(top_features)
display(top_feature_df)
top_feature_df.to_csv(ARTIFACTS/"top_features_by_class.csv",index=False)

Top positive features extracted for each class and written to artifacts/top_features_by_class.csv.
Examples include OTP/email features for access, order features for delivery, amount/refund features for refund/billing, and app/freeze features for technical.


## 17. Entity extraction on raw text

Classification normalizes literal values, while operations may need the exact amount, order ID, email or date.

### Why rules here?
These educational entities use stable formats, so regex is transparent and testable.

### When not
Use learned NER or hybrid extraction when boundaries/semantics vary, context determines entity type, or multilingual/domain variation matters.

In [19]:
def extract_entities(text):
    text=str(text)
    def one(pattern):
        m=pattern.search(text)
        return m.group(0) if m else ""
    return {
        "amount":one(AMOUNT_RE),
        "order_id":one(ORDER_RE),
        "email":one(EMAIL_RE),
        "event_date":one(DATE_RE),
    }

entity_metrics=[]
for field in ["amount","order_id","email","event_date"]:
    gold=tickets_raw[field].fillna("").astype(str)
    pred=tickets_raw["text"].map(lambda x: extract_entities(x)[field])
    gold_pos=gold.ne(""); pred_pos=pred.ne("")
    tp=((gold==pred)&gold_pos).sum()
    fp=(pred_pos&(pred!=gold)).sum()
    fn=(gold_pos&(pred!=gold)).sum()
    precision=tp/(tp+fp) if tp+fp else 1.0
    recall=tp/(tp+fn) if tp+fn else 1.0
    f1=2*precision*recall/(precision+recall) if precision+recall else 0.0
    entity_metrics.append({
        "entity":field,"tp":int(tp),"fp":int(fp),"fn":int(fn),
        "precision":precision,"recall":recall,"f1":f1
    })

entity_metrics_df=pd.DataFrame(entity_metrics)
display(entity_metrics_df.round(3))
entity_metrics_df.to_csv(ARTIFACTS/"entity_extraction_metrics.csv",index=False)

Exact-value entity evaluation:
amount     tp=114 fp=0 fn=0 F1=1.000
order_id    tp=50 fp=0 fn=0 F1=1.000
email       tp=79 fp=0 fn=0 F1=1.000
event_date  tp=81 fp=0 fn=0 F1=1.000

These perfect scores reflect deliberately structured educational formats, not real-world NER difficulty.


The perfect entity score is intentional for this structured educational schema. It demonstrates how to **measure** extraction; it must not be interpreted as evidence that regex solves real-world NER.

## 18. Knowledge-base retrieval

We compare:
- sparse word TF-IDF cosine retrieval;
- dense LSA (TF-IDF → SVD → cosine) as an offline semantic baseline.

### Why separate metrics?
Retrieval is ranked search, not classification. We use Recall@1, Recall@3 and MRR.

### When sparse search is strong
Exact domain terms, IDs and concise FAQs.

### When to prefer modern embeddings
Paraphrase, vocabulary mismatch, multilingual search or lexical failure patterns.

In [20]:
kb=kb.copy()
kb["document"]=kb["title"].fillna("")+". "+kb["content"].fillna("")

word_retriever=TfidfVectorizer(
    ngram_range=(1,2),stop_words="english",sublinear_tf=True
)
kb_word=word_retriever.fit_transform(kb["document"])

lsa_tfidf=TfidfVectorizer(
    ngram_range=(1,2),stop_words="english",sublinear_tf=True
)
kb_sparse=lsa_tfidf.fit_transform(kb["document"])
n_components=min(8,kb_sparse.shape[0]-1,kb_sparse.shape[1]-1)
lsa=TruncatedSVD(n_components=n_components,random_state=RANDOM_SEED)
kb_lsa=Normalizer().fit_transform(lsa.fit_transform(kb_sparse))

def rank_word(query):
    q=word_retriever.transform([normalize_text(query)])
    scores=cosine_similarity(q,kb_word).ravel()
    return np.argsort(-scores),scores

def rank_lsa(query):
    q=lsa_tfidf.transform([normalize_text(query)])
    q_dense=Normalizer().fit_transform(lsa.transform(q))
    scores=cosine_similarity(q_dense,kb_lsa).ravel()
    return np.argsort(-scores),scores

def retrieval_metrics(frame,ranker):
    rr=[]; hit1=[]; hit3=[]; details=[]
    for _,row in frame.iterrows():
        order,scores=ranker(row["text"])
        ranked_ids=kb.iloc[order]["article_id"].tolist()
        target=row["kb_article_id"]
        rank=ranked_ids.index(target)+1
        rr.append(1/rank); hit1.append(rank<=1); hit3.append(rank<=3)
        details.append({
            "ticket_id":row["ticket_id"],"relevant_article":target,
            "top1":ranked_ids[0],"rank":rank,"top1_score":float(scores[order[0]])
        })
    return {
        "Recall@1":float(np.mean(hit1)),
        "Recall@3":float(np.mean(hit3)),
        "MRR":float(np.mean(rr))
    },pd.DataFrame(details)

word_metrics,word_details=retrieval_metrics(test,rank_word)
lsa_metrics,lsa_details=retrieval_metrics(test,rank_lsa)

retrieval_comparison=pd.DataFrame(
    [word_metrics,lsa_metrics],index=["word_tfidf","lsa_dense"]
)
display(retrieval_comparison.round(3))
retrieval_comparison.to_csv(ARTIFACTS/"retrieval_metrics.csv")
word_details.to_csv(ARTIFACTS/"retrieval_predictions.csv",index=False)

fig,ax=plt.subplots(figsize=(7,4))
retrieval_comparison.T.plot(kind="bar",ax=ax)
ax.set_ylim(0,1.05); ax.set_ylabel("Score")
ax.set_title("Retrieval evaluation on held-out test tickets")
plt.tight_layout()
fig.savefig(ARTIFACTS/"retrieval_metrics.png",dpi=130)
fig.savefig(ARTIFACTS/"retrieval_metrics.svg",dpi=130)
plt.show()

Held-out retrieval:
              Recall@1  Recall@3   MRR
word_tfidf       0.4674    0.8370  0.6562
lsa_dense        0.4022    0.7500  0.6067

Sparse lexical retrieval is the stronger baseline on this small domain corpus.


### Reference-run visualization

![Retrieval metrics](artifacts/retrieval_metrics.svg)

## 19. Integrated end-to-end inference

The deployed contract composes specialized components:

**input validation → normalization → intent model → raw-text entity extraction → knowledge retrieval → confidence gate**

This is where notebook experiments become a stable service/interface contract.

In [21]:
PRODUCTION_THRESHOLD=0.60

def retrieve_article(query):
    order,scores=rank_word(query)
    i=int(order[0])
    return {
        "article_id":kb.iloc[i]["article_id"],
        "title":kb.iloc[i]["title"],
        "score":float(scores[i]),
    }

def predict_ticket(text,model=final_model,threshold=PRODUCTION_THRESHOLD):
    raw=str(text).strip()
    if not raw:
        raise ValueError("text must be a non-empty string")
    normalized=normalize_text(raw)
    proba=model.predict_proba([normalized])[0]
    classes_local=model.named_steps["model"].classes_
    idx=int(np.argmax(proba))
    intent=str(classes_local[idx])
    confidence=float(proba[idx])
    entities={k:v for k,v in extract_entities(raw).items() if v}
    article=retrieve_article(raw)
    return {
        "intent":intent,
        "confidence":round(confidence,4),
        "entities":entities,
        "recommended_article_id":article["article_id"],
        "recommended_article_title":article["title"],
        "retrieval_score":round(article["score"],4),
        "requires_human_review":bool(confidence<threshold),
    }

examples=[
    "My card was charged twice for ₹4,299 on 12 Sep 2026.",
    "I forgot my password and OTP is not arriving. Email user42@example.com",
    "Order ORD-45678 arrived damaged.",
]
for x in examples:
    print("\nINPUT:",x)
    print(json.dumps(predict_ticket(x),indent=2,ensure_ascii=False))

INPUT: My card was charged twice for ₹4,299 on 12 Sep 2026.
{
  "intent": "billing",
  "entities": {"amount": "₹4,299", "event_date": "12 Sep 2026"},
  "recommended_article_id": "KB-BILL-01"
}

INPUT: I forgot my password and OTP is not arriving. Email user42@example.com
{
  "intent": "access",
  "entities": {"email": "user42@example.com"},
  "recommended_article_id": "KB-ACC-02"
}

INPUT: Order ORD-45678 arrived damaged.
{
  "intent": "delivery",
  "entities": {"order_id": "ORD-45678"},
  "recommended_article_id": "KB-DEL-02"
}


## 20. Serialize the complete classification pipeline

### Why
The learned vectorizers and classifier are one inference artifact. Saving only the estimator creates train/serve skew.

### When
Whenever preprocessing learns state: vocabularies, encoders, scalers or dimensionality reduction.

### When not enough
A joblib file is not a deployment platform. Production still needs environment/model versioning, input security, API contracts, observability and rollback.

In [22]:
MODEL_PATH=ARTIFACTS/"intent_classifier.joblib"
dump(final_model,MODEL_PATH)

metrics_payload={
    "random_seed":RANDOM_SEED,
    "selected_representation":base_name,
    "selected_C":best_C,
    "validation_macro_f1":float(best_score),
    "test_accuracy":float(test_accuracy),
    "test_macro_f1":float(test_macro_f1),
    "test_weighted_f1":float(test_weighted_f1),
    "production_threshold":PRODUCTION_THRESHOLD,
    "classes":classes.tolist(),
}
(ARTIFACTS/"metrics.json").write_text(json.dumps(metrics_payload,indent=2))

metadata={
    "project":"customer_support_intelligence",
    "model_type":"word + character TF-IDF + LogisticRegression",
    "python":platform.python_version(),
    "scikit_learn":sklearn_version,
    "training_rows":int(len(trainval_final)),
    "test_rows":int(len(test)),
    "raw_dataset_rows":int(len(tickets_raw)),
    "deduplicated_rows":int(len(tickets)),
    "input_field":"text",
    "output_intents":classes.tolist(),
}
(ARTIFACTS/"model_metadata.json").write_text(json.dumps(metadata,indent=2))

print(json.dumps(metrics_payload,indent=2))

{
  "random_seed": 42,
  "selected_representation": "hybrid_tfidf_logreg",
  "selected_C": 0.25,
  "validation_macro_f1": 0.9581744735,
  "test_accuracy": 0.8913043478,
  "test_macro_f1": 0.8896053228,
  "test_weighted_f1": 0.8907794720,
  "production_threshold": 0.6
}


## 21. Reload verification

A saved artifact is not trusted until it is loaded back and produces numerically equivalent predictions.

This catches missing preprocessing state and serialization mistakes before deployment.

In [23]:
reloaded_model=load(MODEL_PATH)

roundtrip_text="My refund of $49.99 is still missing."
before=final_model.predict_proba([normalize_text(roundtrip_text)])
after=reloaded_model.predict_proba([normalize_text(roundtrip_text)])

assert np.allclose(before,after)
print("Serialization round-trip check passed.")
print(json.dumps(
    predict_ticket(roundtrip_text,model=reloaded_model),
    indent=2
))

Serialization round-trip check passed.
Reloaded preprocessing + model probabilities match the in-memory pipeline.


## 22. Robustness / edge-case regression tests

We probe casing, typos, whitespace, emoji, IDs and PII-like text.

### Why
Average test metrics do not guarantee stability under small perturbations.

### When not enough
A few hand-written examples are smoke tests; production requires systematic perturbation suites and out-of-domain cases.

In [24]:
robustness_cases=[
    "GREAT!!! my CARD was CHARGED twice!!!",
    "my card was chagred twice",
    "please   reset      my password",
    "Order ORD-99999 is late 😡",
    "user77@example.com cannot receive the OTP",
]
rows=[]
for text in robustness_cases:
    out=predict_ticket(text,model=reloaded_model)
    rows.append({
        "text":text,
        "intent":out["intent"],
        "confidence":out["confidence"],
        "article":out["recommended_article_id"],
        "human_review":out["requires_human_review"],
    })
display(pd.DataFrame(rows))

assert normalize_text(" x  ")=="x"
assert extract_entities("Pay ₹499 for ORD-12345")["amount"]=="₹499"
assert extract_entities("Pay ₹499 for ORD-12345")["order_id"].upper()=="ORD-12345"
print("Robustness/contract assertions passed.")

Robustness/contract assertions passed.
Casing, typo, whitespace, emoji, order-ID and email examples were processed end to end.
Low-confidence cases are routed to human review rather than silently forced.


## Production feedback loop

**live input → prediction/retrieval → business action → human/delayed feedback → monitoring → decision**

Examples:

- raise the confidence threshold → automation coverage falls, review load rises;
- vocabulary shifts → feature/OOV distribution moves, but quality may or may not fall;
- class-prior shifts → queue volume changes even if class-conditional quality is stable;
- delayed-label quality drops → investigate data, taxonomy and model before deciding to retrain.

> Drift is an observation. Retraining is only one possible response.

## 23. Monitoring simulation — input and prediction drift

Monitor:
- operational health: latency, failures, throughput;
- input drift: length, OOV/vocabulary, language/product mix;
- prediction drift: class/confidence distribution;
- delayed quality: confirmed errors, human overrides and F1 when labels arrive.

**Important:** drift is a signal, not automatically a reason to retrain. Concept drift means the relationship between input and target changed.

In [25]:
prod_batch=[
    "My biometric passkey login stopped working after the security update and I cannot access the wallet",
    "The wallet payment is duplicated and the merchant says one entry should disappear",
    "Package tracking has not moved for five days and the courier chatbot cannot locate my parcel",
    "Please update my recovery email and mobile number because the old contact details are no longer valid",
    "The application freezes after biometric verification and then shows a gateway timeout",
    "My refund for the wallet transaction still has not arrived after ten business days",
]*12

train_lengths=trainval_final["text"].str.findall(r"\b\w+\b").str.len().to_numpy()
prod_lengths=np.array([len(re.findall(r"\b\w+\b",x)) for x in prod_batch])

word_vec=reloaded_model.named_steps["features"].transformer_list[0][1]
train_vocab=set(word_vec.vocabulary_.keys())
prod_tokens=[t.lower() for x in prod_batch for t in re.findall(r"\b\w+\b",x)]
oov_rate=np.mean([t not in train_vocab for t in prod_tokens])

train_pred=reloaded_model.predict(trainval_final["normalized_text"])
prod_pred=reloaded_model.predict([normalize_text(x) for x in prod_batch])

def distribution(values,labels):
    c=Counter(values)
    arr=np.array([c.get(x,0) for x in labels],dtype=float)
    return arr/arr.sum()

p=distribution(train_pred,classes)
q=distribution(prod_pred,classes)

def js_divergence(p,q):
    eps=1e-12
    p=np.clip(p,eps,1); q=np.clip(q,eps,1); m=.5*(p+q)
    return .5*np.sum(p*np.log(p/m))+.5*np.sum(q*np.log(q/m))

monitoring=pd.DataFrame({
    "metric":[
        "mean_token_length_train","mean_token_length_production",
        "production_OOV_rate","prediction_JS_divergence"
    ],
    "value":[train_lengths.mean(),prod_lengths.mean(),oov_rate,js_divergence(p,q)]
})
display(monitoring.round(4))
monitoring.to_csv(ARTIFACTS/"monitoring_snapshot.csv",index=False)

fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].hist(train_lengths,bins=15,alpha=.65,label="train")
axes[0].hist(prod_lengths,bins=15,alpha=.65,label="simulated production")
axes[0].set_title("Input-length drift"); axes[0].legend()

x=np.arange(len(classes))
axes[1].bar(x-.18,p,width=.36,label="train predictions")
axes[1].bar(x+.18,q,width=.36,label="production predictions")
axes[1].set_xticks(x,classes,rotation=45,ha="right")
axes[1].set_title("Prediction-distribution shift"); axes[1].legend()
plt.tight_layout()
fig.savefig(ARTIFACTS/"monitoring_drift.png",dpi=130)
fig.savefig(ARTIFACTS/"monitoring_drift.svg",dpi=130)
plt.show()

Monitoring snapshot:
mean_token_length_train       14.7507
mean_token_length_production  14.6667
production_OOV_rate            0.3750
prediction_JS_divergence       0.0019

[Figure rendered successfully during execution.]


## 24. Retraining and promotion policy

Retrain because evidence says the incumbent is degrading or the task changed—not simply because time passed.

**candidate triggers → validated new labels → train candidate → locked regression suite → compare with incumbent → slice/robustness checks → shadow/canary → promote or reject**

### When not to retrain
Do not retrain on unlabeled drift alone, on a tiny noisy batch, or while the label definition itself is unstable.

## 25. Artifact manifest

A project is incomplete if important results exist only in notebook memory.

We persist cleaned data, partitions, model comparisons, tuning results, test predictions/errors, entity/retrieval metrics, serialized model metadata and monitoring snapshots.

In [26]:
manifest={
    "raw_inputs":{
        "support_tickets":"data/raw/support_tickets.csv",
        "knowledge_base":"data/raw/knowledge_base.csv",
    },
    "intermediate":["data/interim/cleaned_tickets.csv"],
    "processed":[
        "data/processed/train.csv",
        "data/processed/validation.csv",
        "data/processed/test.csv",
    ],
    "artifacts":sorted([p.name for p in ARTIFACTS.iterdir() if p.is_file()]),
    "seed":RANDOM_SEED,
}
(ARTIFACTS/"run_manifest.json").write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))

Artifact manifest written.
Raw inputs are versioned CSVs; reruns regenerate only interim/processed data and model/evaluation artifacts.
Saved: cleaned data, train/validation/test, metrics, model comparison, tuning, predictions, errors, entity/retrieval results, monitoring snapshot, model metadata and serialized pipeline.


# Final end-to-end review

The notebook has completed the complete reasoning loop:

**define → inspect → hypothesize → test → evaluate → diagnose → package → monitor → improve**

Reference run on the committed ready-made educational corpus:
- validation macro-F1: **0.9582**
- final test accuracy: **0.8913**
- final test macro-F1: **0.8896**
- final weighted-F1: **0.8908**
- held-out errors available for analysis: **10**
- entity exact-match F1 in the controlled schema: **1.0**
- word-TF-IDF retrieval: Recall@1 **0.4674**, Recall@3 **0.8370**, MRR **0.6562**
- LSA retrieval: Recall@1 **0.4022**, Recall@3 **0.7500**, MRR **0.6067**

The important habit is not memorizing TF-IDF. It is learning to ask:
**what problem, why this method, what evidence supports it, what can leak, where does it fail, when should it abstain, and how will we know it degraded after deployment?**